# F1 Visor · Experiment review

Training lives in `train.py`; this notebook displays files from an actual new run. Historical saved notebook scores are documented separately in `docs/HISTORY.md`. Install `requirements-notebook.txt` and launch Jupyter from the repository root.

## Reproduce from the terminal
```bash
python prepare_data.py --source data/raw --output data/split
python train.py --data data/split --output runs/experiment-01
python export_model.py --checkpoint runs/experiment-01/best.pt
```
These commands train on your own `clean/` and `stain/` images; pretrained ImageNet weights download on the first run.

In [ ]:
from pathlib import Path
import csv, json
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "train.py").exists() else Path.cwd().parent
RUN = ROOT / "runs" / "experiment-01"
if not (RUN / "metrics.json").is_file():
    raise FileNotFoundError("Complete the training commands above, or select an existing RUN")
report = json.loads((RUN / "metrics.json").read_text())
report

In [ ]:
with (RUN / "history.csv").open() as f:
    history = list(csv.DictReader(f))
epochs = [int(row["epoch"]) for row in history]
fig, ax = plt.subplots(figsize=(8, 4))
for name in ("train_loss", "val_loss"):
    ax.plot(epochs, [float(row[name]) for row in history], label=name)
ax.set(xlabel="Epoch", ylabel="Loss", title="Training history")
ax.legend()
plt.show()

In [ ]:
matrix = report["test"]["confusion_matrix"]
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(matrix, cmap="Blues")
for row in range(2):
    for col in range(2):
        ax.text(col, row, str(matrix[row][col]), ha="center", va="center")
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["clean", "stain"], yticklabels=["clean", "stain"], xlabel="Predicted", ylabel="Actual", title="Held-out test confusion matrix")
plt.show()

## Interpretation
The threshold is chosen on validation data; the test set is evaluated only after model selection. Report dataset provenance, camera conditions, image size, class counts and device when presenting a score. A synthetic smoke test verifies software behavior, not recognition quality. Original notebook files remain with the author; this review notebook replaces duplicated exploratory cells.

## Full evaluation report

After a current training run, generate PR/ROC, confusion matrices, probability distributions and hard examples from the terminal:
```bash
python evaluation_plots.py --run runs/experiment-01 --data data/split
```
The CLI reads `predictions.csv` and the saved validation threshold. It does not recompute a threshold from test labels. For image collection and manual labeling, see `docs/COLLECTION.md`.